# Interpretability + Debate Experiments —  7

This notebook adds two new experiments based on prof.Ying Ding feedback:

1. **Interpretability + Faithfulness** — instead of just asking the LLM to predict 0/1, I ask it to also explain its reasoning. Then I check if that explanation matches what SHAP says actually drove the XGBoost prediction. This evaluates whether LLM reasoning is trustworthy, not just whether its predictions are correct.

2. **LLM Debate on Borderline Cases** — for the 12 patients the LLM consistently got wrong (PHQ 8-9), I run a structured debate: one LLM argues high risk, another argues low risk, a third acts as judge. The question: can debate resolve uncertainty on borderline cases?

In [2]:
# Setup
from google.colab import drive
drive.mount('/content/drive')

!pip install groq -q

import pandas as pd
import numpy as np
import pickle
import re
import time
from groq import Groq
from sklearn.model_selection import train_test_split

base = '/content/drive/MyDrive/PTSD_Project'
processed = base + '/data/processed'
results = base + '/results'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Load data and predictions
with open(results + '/all_predictions.pkl', 'rb') as f:
    all_preds = pickle.load(f)

y_sample = pd.Series(all_preds['y_sample'])
sample_idx = all_preds['sample_idx']

# reload original test rows
merged = pd.read_csv(processed + '/merged_clean.csv')
y = merged['label']
_, X_test_orig, _, _ = train_test_split(
    merged, y, test_size=0.2, random_state=42, stratify=y
)
X_test_orig = X_test_orig.reset_index(drop=True)
X_sample = X_test_orig.iloc[sample_idx].reset_index(drop=True)

# load SHAP top features
top_features = pd.read_csv(results + '/top_features_with_values.csv')
top_features_sample = top_features.iloc[sample_idx].reset_index(drop=True)

# add PHQ total to sample
phq_cols = ['DPQ010','DPQ020','DPQ030','DPQ040','DPQ050',
            'DPQ060','DPQ070','DPQ080','DPQ090']
X_sample['PHQ_total'] = X_sample[phq_cols].sum(axis=1)

print("loaded!")
print("sample size:", len(y_sample))
print("high risk in sample:", y_sample.sum())
print("borderline (PHQ 8-9):", ((X_sample['PHQ_total'] >= 8) & (X_sample['PHQ_total'] <= 9)).sum())

loaded!
sample size: 200
high risk in sample: 15
borderline (PHQ 8-9): 12


In [4]:
# Groq setup
GROQ_API_KEY = "GROQ_API_KEY"

client = Groq(api_key=GROQ_API_KEY)

def call_groq(prompt, max_tokens=200):
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=max_tokens
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        time.sleep(3)
        return "error"

# test
print(call_groq("working", max_tokens=10))

It seems like you're in work mode. What


In [5]:
# Get high risk patients
high_risk_idx = [i for i in range(200) if y_sample.iloc[i] == 1]
borderline_idx = [i for i in range(200)
                  if X_sample.iloc[i]['PHQ_total'] >= 8
                  and X_sample.iloc[i]['PHQ_total'] <= 9
                  and y_sample.iloc[i] == 0]

print("High risk patients:", len(high_risk_idx))
print("Borderline patients (PHQ 8-9, low risk label):", len(borderline_idx))

# quick look at high risk patients
hr_df = X_sample.iloc[high_risk_idx][['RIDAGEYR', 'RIAGENDR', 'PHQ_total', 'SLD012']].copy()
hr_df['y_true'] = y_sample.iloc[high_risk_idx].values
print("\nHigh risk patient profiles:")
print(hr_df.to_string())

High risk patients: 15
Borderline patients (PHQ 8-9, low risk label): 12

High risk patient profiles:
     RIDAGEYR  RIAGENDR  PHQ_total  SLD012  y_true
32       71.0       1.0         10    12.0       1
37       62.0       2.0         17     8.0       1
39       26.0       2.0         16     3.0       1
55       57.0       2.0         11     7.5       1
59       64.0       1.0         20     6.0       1
63       64.0       2.0         10     7.0       1
79       67.0       2.0         10    10.0       1
83       40.0       1.0         25     8.0       1
98       59.0       2.0         10     7.5       1
104      32.0       2.0         22     4.5       1
106      24.0       2.0         10     8.5       1
141      21.0       2.0         12     6.0       1
145      39.0       2.0         10     8.0       1
169      60.0       1.0         10     4.0       1
186      21.0       2.0         10     6.0       1


In [6]:
# Interpretability experiment
# Ask LLM to predict AND explain reasoning for high risk patients

def get_interpretation(row, shap_row):
    gender = "female" if row['RIAGENDR'] == 2.0 else "male"
    phq = int(row['PHQ_total'])

    prompt = f"""You are a clinical assistant helping assess depression/PTSD risk.

Patient data:
- Age: {int(row['RIDAGEYR'])}, Sex: {gender}
- PHQ-9 total score: {phq}
- Sleep hours per night: {row['SLD012']}
- Alcohol days per year: {int(row['ALQ121'])}
- Income to poverty ratio: {round(row['INDFMPIR'], 1)}

Individual PHQ-9 responses (0=not at all, 3=nearly every day):
- Little interest/pleasure (DPQ010): {int(row['DPQ010'])}
- Feeling down/depressed (DPQ020): {int(row['DPQ020'])}
- Sleep problems (DPQ030): {int(row['DPQ030'])}
- Feeling tired (DPQ040): {int(row['DPQ040'])}
- Poor appetite (DPQ050): {int(row['DPQ050'])}
- Feeling bad about yourself (DPQ060): {int(row['DPQ060'])}
- Trouble concentrating (DPQ070): {int(row['DPQ070'])}
- Moving slowly/restless (DPQ080): {int(row['DPQ080'])}
- Thoughts of self-harm (DPQ090): {int(row['DPQ090'])}

Task:
1. Predict risk: 0 (low) or 1 (high)
2. In 2-3 sentences explain which specific symptoms drove your prediction.

Format your response exactly as:
PREDICTION: [0 or 1]
EXPLANATION: [your explanation]"""

    return call_groq(prompt, max_tokens=150)

# run on all 15 high risk patients
print("Running interpretability experiment...")
interp_results = []

for i, idx in enumerate(high_risk_idx):
    row = X_sample.iloc[idx]
    shap_row = top_features_sample.iloc[idx]

    response = get_interpretation(row, shap_row)

    interp_results.append({
        'patient_idx': idx,
        'PHQ_total': int(row['PHQ_total']),
        'shap_top1': shap_row['Top1'],
        'shap_top2': shap_row['Top2'],
        'shap_top3': shap_row['Top3'],
        'llm_response': response,
        'y_true': int(y_sample.iloc[idx])
    })

    print(f"Patient {i+1}/15 done")
    time.sleep(1.5)

interp_df = pd.DataFrame(interp_results)
print("\ndone")
print(interp_df[['patient_idx', 'PHQ_total', 'llm_response']].to_string())

Running interpretability experiment...
Patient 1/15 done
Patient 2/15 done
Patient 3/15 done
Patient 4/15 done
Patient 5/15 done
Patient 6/15 done
Patient 7/15 done
Patient 8/15 done
Patient 9/15 done
Patient 10/15 done
Patient 11/15 done
Patient 12/15 done
Patient 13/15 done
Patient 14/15 done
Patient 15/15 done

done
    patient_idx  PHQ_total                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   llm_response
0            32         10                                                                   

In [7]:
# Parse predictions and evaluate faithfulness

# PHQ question mapping for faithfulness check
phq_mapping = {
    'DPQ010': ['interest', 'pleasure', 'little interest'],
    'DPQ020': ['down', 'depressed', 'hopeless'],
    'DPQ030': ['sleep', 'sleeping', 'insomnia'],
    'DPQ040': ['tired', 'fatigue', 'energy'],
    'DPQ050': ['appetite', 'eating', 'weight'],
    'DPQ060': ['bad about', 'worthless', 'failure', 'self-criticism'],
    'DPQ070': ['concentrat', 'focus'],
    'DPQ080': ['moving slowly', 'restless', 'slow'],
    'DPQ090': ['self-harm', 'suicidal', 'better off dead']
}

def parse_response(response):
    pred = -1
    explanation = ""
    if 'PREDICTION:' in response:
        pred_part = response.split('PREDICTION:')[1].split('\n')[0].strip()
        if '0' in pred_part:
            pred = 0
        elif '1' in pred_part:
            pred = 1
    if 'EXPLANATION:' in response:
        explanation = response.split('EXPLANATION:')[1].strip()
    return pred, explanation

def check_faithfulness(explanation, shap_top1, shap_top2, shap_top3):
    explanation_lower = explanation.lower()
    shap_features = [shap_top1.split('=')[0],
                     shap_top2.split('=')[0],
                     shap_top3.split('=')[0]]

    matched = []
    for feature in shap_features:
        keywords = phq_mapping.get(feature, [])
        for keyword in keywords:
            if keyword.lower() in explanation_lower:
                matched.append(feature)
                break

    faithfulness_score = len(matched) / len(shap_features)
    return faithfulness_score, matched

# analyze all results
analysis_rows = []
for _, row in interp_df.iterrows():
    pred, explanation = parse_response(row['llm_response'])
    faith_score, matched = check_faithfulness(
        explanation, row['shap_top1'], row['shap_top2'], row['shap_top3']
    )

    correct = (pred == row['y_true'])
    right_wrong_reasons = correct and faith_score < 0.5

    analysis_rows.append({
        'patient_idx': row['patient_idx'],
        'PHQ_total': row['PHQ_total'],
        'y_true': row['y_true'],
        'llm_pred': pred,
        'correct': correct,
        'explanation': explanation,
        'shap_top1': row['shap_top1'],
        'shap_top2': row['shap_top2'],
        'shap_top3': row['shap_top3'],
        'matched_features': str(matched),
        'faithfulness_score': round(faith_score, 2),
        'right_wrong_reasons': right_wrong_reasons
    })

analysis_df = pd.DataFrame(analysis_rows)

print("=== Interpretability Results ===\n")
print(f"Total high risk patients: {len(analysis_df)}")
print(f"LLM correct predictions: {analysis_df['correct'].sum()}/{len(analysis_df)}")
print(f"Average faithfulness score: {analysis_df['faithfulness_score'].mean():.2f}")
print(f"Right for wrong reasons: {analysis_df['right_wrong_reasons'].sum()}")

print("\n--- Per patient summary ---")
cols = ['patient_idx', 'PHQ_total', 'llm_pred', 'correct',
        'faithfulness_score', 'right_wrong_reasons']
print(analysis_df[cols].to_string(index=False))

=== Interpretability Results ===

Total high risk patients: 15
LLM correct predictions: 9/15
Average faithfulness score: 0.69
Right for wrong reasons: 2

--- Per patient summary ---
 patient_idx  PHQ_total  llm_pred  correct  faithfulness_score  right_wrong_reasons
          32         10         1     True                1.00                False
          37         17         1     True                1.00                False
          39         16         1     True                1.00                False
          55         11         0    False                0.67                False
          59         20         1     True                1.00                False
          63         10         0    False                0.67                False
          79         10         1     True                0.67                False
          83         25         1     True                0.67                False
          98         10         1     True                0.33

In [9]:
# Save interpretability results
analysis_df.to_csv(results + '/interpretability_results.csv', index=False)
print("saved")

# summary stats for report
print("\n=== Key Stats for Report ===")
print(f"Accuracy on high risk patients: {analysis_df['correct'].mean():.2f}")
print(f"Average faithfulness: {analysis_df['faithfulness_score'].mean():.2f}")
print(f"High faithfulness (>=0.67): {(analysis_df['faithfulness_score'] >= 0.67).sum()}/{len(analysis_df)}")
print(f"Low faithfulness (<0.5): {(analysis_df['faithfulness_score'] < 0.5).sum()}/{len(analysis_df)}")
print(f"Right for wrong reasons: {analysis_df['right_wrong_reasons'].sum()}/{len(analysis_df)}")
print(f"\nPattern: LLM fails on PHQ 10-12, succeeds on PHQ 16+")
print(f"Low PHQ correct: {analysis_df[analysis_df['PHQ_total']<=12]['correct'].sum()}/{len(analysis_df[analysis_df['PHQ_total']<=12])}")
print(f"High PHQ correct: {analysis_df[analysis_df['PHQ_total']>12]['correct'].sum()}/{len(analysis_df[analysis_df['PHQ_total']>12])}")

saved

=== Key Stats for Report ===
Accuracy on high risk patients: 0.60
Average faithfulness: 0.69
High faithfulness (>=0.67): 11/15
Low faithfulness (<0.5): 4/15
Right for wrong reasons: 2/15

Pattern: LLM fails on PHQ 10-12, succeeds on PHQ 16+
Low PHQ correct: 4/10
High PHQ correct: 5/5


## Experiment 2 — LLM Debate on Borderline Patients

The 12 patients with PHQ scores of 8-9 failed consistently across all prompt formats in notebook 6. Here I try a different approach — structured debate between two LLM instances — to see if adversarial reasoning can resolve these borderline cases.

In [11]:
# Debate experiment on borderline patients

def run_debate(row):
    gender = "female" if row['RIAGENDR'] == 2.0 else "male"
    phq = int(row['PHQ_total'])

    patient_desc = f"""Patient: {int(row['RIDAGEYR'])}-year-old {gender}
PHQ-9 total: {phq}
Individual symptoms (0=not at all, 3=nearly every day):
- Little interest/pleasure: {int(row['DPQ010'])}
- Feeling down/depressed: {int(row['DPQ020'])}
- Sleep problems: {int(row['DPQ030'])}
- Feeling tired: {int(row['DPQ040'])}
- Poor appetite: {int(row['DPQ050'])}
- Feeling bad about self: {int(row['DPQ060'])}
- Trouble concentrating: {int(row['DPQ070'])}
- Moving slowly/restless: {int(row['DPQ080'])}
- Thoughts of self-harm: {int(row['DPQ090'])}
- Sleep hours: {row['SLD012']}
- Income ratio: {round(row['INDFMPIR'], 1)}"""

    # Argument FOR high risk
    prompt_for = f"""{patient_desc}

You are a clinical expert arguing that this patient IS at high risk for depression/PTSD.
Give 2-3 specific reasons based on the symptoms above why this patient should be considered high risk.
Be concise."""

    # Argument AGAINST high risk
    prompt_against = f"""{patient_desc}

You are a clinical expert arguing that this patient is NOT at high risk for depression/PTSD.
Give 2-3 specific reasons based on the symptoms above why this patient should be considered low risk.
Be concise."""

    arg_for = call_groq(prompt_for, max_tokens=150)
    time.sleep(1.5)
    arg_against = call_groq(prompt_against, max_tokens=150)
    time.sleep(1.5)

    # Judge makes final decision
    prompt_judge = f"""{patient_desc}

Two clinical experts have reviewed this patient:

Expert A (argues HIGH risk): {arg_for}

Expert B (argues LOW risk): {arg_against}

As the final judge, carefully consider both arguments and make a final decision.
Reply in this exact format:
VERDICT: [0 for low risk or 1 for high risk]
REASON: [one sentence explaining your decision]"""

    verdict_response = call_groq(prompt_judge, max_tokens=100)
    time.sleep(1.5)

    return arg_for, arg_against, verdict_response

print("Running debate experiment on 12 borderline patients...")
debate_results = []

for i, idx in enumerate(borderline_idx):
    row = X_sample.iloc[idx]

    arg_for, arg_against, verdict = run_debate(row)

    # parse verdict
    verdict_pred = -1
    if 'VERDICT:' in verdict:
        v = verdict.split('VERDICT:')[1].split('\n')[0].strip()
        if '0' in v:
            verdict_pred = 0
        elif '1' in v:
            verdict_pred = 1

    debate_results.append({
        'patient_idx': idx,
        'PHQ_total': int(row['PHQ_total']),
        'y_true': int(y_sample.iloc[idx]),
        'verdict_pred': verdict_pred,
        'correct': verdict_pred == int(y_sample.iloc[idx]),
        'arg_for': arg_for,
        'arg_against': arg_against,
        'verdict': verdict
    })

    print(f"Patient {i+1}/12 done")

debate_df = pd.DataFrame(debate_results)
print("\ndone")

Running debate experiment on 12 borderline patients...
Patient 1/12 done
Patient 2/12 done
Patient 3/12 done
Patient 4/12 done
Patient 5/12 done
Patient 6/12 done
Patient 7/12 done
Patient 8/12 done
Patient 9/12 done
Patient 10/12 done
Patient 11/12 done
Patient 12/12 done

done


In [12]:
# Analyze debate results
print("=== Debate Experiment Results ===\n")
print(f"Total borderline patients: {len(debate_df)}")
print(f"Correct verdicts: {debate_df['correct'].sum()}/{len(debate_df)}")
print(f"Accuracy: {debate_df['correct'].mean():.2f}")

print("\n--- Per patient results ---")
cols = ['patient_idx', 'PHQ_total', 'y_true', 'verdict_pred', 'correct']
print(debate_df[cols].to_string(index=False))

# compare debate vs single LLM on same patients
# get F1_zero predictions for borderline patients
f1_zero_preds = [all_preds['pred_f1_zero'][i] for i in borderline_idx]
f1_zero_correct = sum(p == int(y_sample.iloc[borderline_idx[i]])
                      for i, p in enumerate(f1_zero_preds))

print(f"\n=== Comparison on borderline patients ===")
print(f"Single LLM (F1_zero) accuracy: {f1_zero_correct}/{len(borderline_idx)}")
print(f"Debate LLM accuracy: {debate_df['correct'].sum()}/{len(debate_df)}")
print(f"\nDid debate help? {'Yes' if debate_df['correct'].sum() > f1_zero_correct else 'No'}")

# save
debate_df.to_csv(results + '/debate_results.csv', index=False)
print("\nsaved")

=== Debate Experiment Results ===

Total borderline patients: 12
Correct verdicts: 0/12
Accuracy: 0.00

--- Per patient results ---
 patient_idx  PHQ_total  y_true  verdict_pred  correct
          17          8       0             1    False
          34          8       0             1    False
          35          9       0             1    False
          40          8       0             1    False
          61          9       0             1    False
          70          8       0             1    False
          94          8       0             1    False
         114          9       0             1    False
         132          9       0             1    False
         140          9       0             1    False
         153          9       0             1    False
         189          9       0             1    False

=== Comparison on borderline patients ===
Single LLM (F1_zero) accuracy: 0/12
Debate LLM accuracy: 0/12

Did debate help? No

saved


In [13]:
# Final summary of both experiments

print("=== FINAL SUMMARY ===\n")

print("--- Experiment 1: Interpretability + Faithfulness ---")
print(f"High risk patients analyzed: 15")
print(f"LLM accuracy on high risk: {analysis_df['correct'].mean():.2f}")
print(f"  - PHQ > 12 accuracy: 5/5 (100%)")
print(f"  - PHQ 10-12 accuracy: 4/10 (40%)")
print(f"Average faithfulness score: {analysis_df['faithfulness_score'].mean():.2f}")
print(f"Right for wrong reasons: {analysis_df['right_wrong_reasons'].sum()}/15")

print("\n--- Experiment 2: Debate on Borderline Cases ---")
print(f"Borderline patients (PHQ 8-9): 12")
print(f"Single LLM accuracy: 0/12")
print(f"Debate LLM accuracy: 0/12")
print(f"Debate improvement: None")
print(f"Key finding: Judge consistently sided with high risk regardless of arguments")

print("\n--- Combined Insight ---")
print("LLM applies clinical conservatism on borderline cases.")
print("This cannot be resolved by debate, few-shot, or SHAP-guided prompting.")
print("Fundamental limitation: LLMs cannot learn hard numerical thresholds")
print("without fine-tuning on labeled clinical data.")

# save summary
with open(results + '/notebook7_summary.txt', 'w') as f:
    f.write("NOTEBOOK 7 SUMMARY\n\n")
    f.write("Interpretability: LLM accuracy 60% on high risk, faithfulness 0.69\n")
    f.write("PHQ>12: 100% accurate. PHQ 10-12: 40% accurate.\n")
    f.write("Right for wrong reasons: 2/15\n\n")
    f.write("Debate: 0/12 correct on borderline patients\n")
    f.write("No improvement over single LLM prompting\n")
    f.write("Judge always sided with high risk on PHQ 8-9 patients\n")

=== FINAL SUMMARY ===

--- Experiment 1: Interpretability + Faithfulness ---
High risk patients analyzed: 15
LLM accuracy on high risk: 0.60
  - PHQ > 12 accuracy: 5/5 (100%)
  - PHQ 10-12 accuracy: 4/10 (40%)
Average faithfulness score: 0.69
Right for wrong reasons: 2/15

--- Experiment 2: Debate on Borderline Cases ---
Borderline patients (PHQ 8-9): 12
Single LLM accuracy: 0/12
Debate LLM accuracy: 0/12
Debate improvement: None
Key finding: Judge consistently sided with high risk regardless of arguments

--- Combined Insight ---
LLM applies clinical conservatism on borderline cases.
This cannot be resolved by debate, few-shot, or SHAP-guided prompting.
Fundamental limitation: LLMs cannot learn hard numerical thresholds
without fine-tuning on labeled clinical data.


## Key Takeaway

Both experiments point to the same conclusion: LLMs apply clinical conservatism on borderline cases and cannot be corrected through prompt engineering alone — whether through better formatting, few-shot examples, SHAP guidance, or debate. Fine-tuning on labeled clinical data would be needed to teach the LLM hard numerical thresholds.